In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

In [35]:
# 設定 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.cuda.get_device_name(0)}")
print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")

CUDA available: True
Current device: NVIDIA GeForce RTX 5070
Compute Capability: (12, 0)


In [55]:
#### TODO 讀取紅酒與白酒訓練資料
data_red = pd.read_csv('winequality-red_train.csv', delimiter=',')
data_red['is_white'] = 0
data_white = pd.read_csv('winequality-white_train.csv', delimiter=',')
data_white['is_white'] = 1
data = pd.concat([data_red, data_white], axis=0)
#### TODO
selected_features = True
if selected_features:
  #### TODO 篩選酒類所需特徵，並進行訓練
  white_features = ['fixed acidity', 'volatile acidity', 'residual sugar', 'chlorides', 'total sulfur dioxide', 'density', 'pH', 'alcohol']
  red_features = ['volatile acidity', 'citric acid', 'chlorides', 'total sulfur dioxide', 'density', 'sulphates', 'alcohol']
  selected_features = list(set(red_features + white_features)) + ['is_white']
  #### TODO
if selected_features:
  data = data[selected_features + ['quality']]

# 分割特徵與標籤
X = data.iloc[:, :-1].values  # 特徵
y = data.iloc[:, -1].values   # 標籤 (Wine Quality)

# 標準化數據
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [56]:
# 切分訓練集與測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 轉換為 PyTorch Tensor
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

# 創建 DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [57]:
# 定義神經網絡模型
class WineQualityNN(nn.Module):
    def __init__(self, input_dim):
        super(WineQualityNN, self).__init__()
        #### TODO 類別確認
        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.2)

        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.2)

        self.fc3 = nn.Linear(64, 1)  # Wine quality 範圍為 0-10
        #### TODO

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        return self.fc3(x)

In [ ]:
# 初始化模型
input_dim = X.shape[1]
model = WineQualityNN(input_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

In [65]:
# 訓練模型
def train_model(model, train_loader, criterion, optimizer, scheduler, epochs=20):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch.float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, LR: {current_lr}")

In [66]:
# 測試模型
def test_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch).squeeze()
            predicted = torch.round(outputs)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

In [67]:
# 執行訓練與測試
train_model(model, train_loader, criterion, optimizer, scheduler, epochs=100)
test_model(model, test_loader)

Epoch 10/100, Loss: 0.9214, LR: 0.001
Epoch 20/100, Loss: 0.8055, LR: 0.001
Epoch 30/100, Loss: 0.7441, LR: 0.0001
Epoch 40/100, Loss: 0.7334, LR: 0.0001
Epoch 50/100, Loss: 0.7059, LR: 0.0001
Epoch 60/100, Loss: 0.6999, LR: 1e-05
Epoch 70/100, Loss: 0.6916, LR: 1e-05
Epoch 80/100, Loss: 0.6736, LR: 1e-05
Epoch 90/100, Loss: 0.6898, LR: 1.0000000000000002e-06
Epoch 100/100, Loss: 0.6920, LR: 1.0000000000000002e-06
Test Accuracy: 54.13%


In [68]:
# 預測新數據並保存到同一個 CSV
def predict_and_save_combined(model, selected_features, files, output_csv):
    results = []
    for file_path, wine_type in files:
        data = pd.read_csv(file_path, delimiter=',')
        data['is_white'] = 1 if wine_type == "white" else 0
        if selected_features:
            data = data[selected_features]
        X_new = scaler.transform(data.values)
        X_new_tensor = torch.tensor(X_new, dtype=torch.float32).to(device)

        with torch.no_grad():
            outputs = model(X_new_tensor).squeeze()
            predicted = torch.round(outputs).clamp(0, 10)

        results.extend([
            {'ID': f"{wine_type}_{i+1}", 'quality': int(pred.cpu().numpy())}
            for i, pred in enumerate(predicted)
        ])

    results_df = pd.DataFrame(results)
    results_df.to_csv(output_csv, index=False)
    print(f"Predictions saved to {output_csv}")

In [69]:
# 預測紅酒與白酒品質，合併輸出至單一 CSV
predict_and_save_combined(model,
  selected_features,
 [("winequality-red_goal.csv", "red"), ("winequality-white_goal.csv", "white")],
                          "winequality_predictions.csv")

Predictions saved to winequality_predictions.csv
